# SDXL Compression Study

Systematic optimization of Stable Diffusion XL.

> Rajat Gupta · Pruna AI Technical Interview · April 2026

**All results are saved to Google Drive — survives disconnects.**

In [ ]:
# ============================================================
# CELL 0: Mount Drive + Clone + Install (run every reconnect)
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

# Persistent results directory on Drive
RESULTS_DIR = '/content/drive/MyDrive/sdxl-optimization-results'
!mkdir -p {RESULTS_DIR}

import os
os.chdir('/content')

REPO_URL = 'https://github.com/rajat116/sdxl-optimization.git'
if not os.path.exists('/content/sdxl-optimization/src'):
    !git clone {REPO_URL}

os.chdir('/content/sdxl-optimization')

# Symlink results dir to Drive so all saves persist
!rm -rf results
!ln -s {RESULTS_DIR} results

!pip install -q -e '.[dev]'
!pip install -q DeepCache
print('\n✅ Setup complete. Results persist in Google Drive.')

In [ ]:
# ============================================================
# CELL 1: Imports + GPU check
# ============================================================
import sys, logging, gc, time, json
import torch
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display, HTML, Image as IPImage

sys.path.insert(0, 'src')

from sdxl_opt.utils import (
    setup_logging, seed_everything, get_gpu_info,
    reset_peak_memory, gpu_peak_memory_gb, EVAL_PROMPTS, timer
)
from sdxl_opt.pipeline import CompressionConfig, load_pipeline, generate_images
from sdxl_opt.evaluate import compute_clip_scores, save_comparison_grid, save_individual_images
from sdxl_opt.pareto import plot_pareto, plot_speedup_bar, plot_memory_comparison, find_pareto_frontier

setup_logging('INFO')
gpu = get_gpu_info()
print(f'\n✅ GPU: {gpu}')

In [ ]:
# ============================================================
# CELL 2: Checkpoint system (loads from Drive)
# ============================================================
CHECKPOINT = 'results/checkpoint.json'

def save_checkpoint(results, section):
    with open(CHECKPOINT, 'w') as f:
        json.dump({'results': results, 'section': section}, f)
    print(f'💾 Saved checkpoint after {section} ({len(results)} configs done) → Google Drive')

def load_checkpoint():
    try:
        with open(CHECKPOINT) as f:
            data = json.load(f)
        print(f'📂 Loaded checkpoint: {data["section"]} ({len(data["results"])} configs done)')
        return data['results'], data['section']
    except FileNotFoundError:
        print('No checkpoint found, starting fresh')
        return [], ''

results, last_section = load_checkpoint()
all_images = {}
prompts = EVAL_PROMPTS[:4]

def already_done(name):
    return any(r['config'] == name for r in results)

print(f'Configs already completed: {[r["config"] for r in results]}')

In [ ]:
# ============================================================
# CELL 3: Benchmark function
# ============================================================
def benchmark_one(config, prompts, n_runs=3, n_warmup=2):
    pipe = load_pipeline(config)
    gen = seed_everything(0)
    for _ in range(n_warmup):
        generate_images(pipe, config, ['warmup'], generator=gen, height=512, width=512)
    torch.cuda.synchronize()

    latencies = []
    last_imgs = []
    for run in range(n_runs):
        gen = seed_everything(42)
        reset_peak_memory()
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        imgs = generate_images(pipe, config, prompts, generator=gen)
        torch.cuda.synchronize()
        per_img = (time.perf_counter() - t0) / len(prompts)
        latencies.append(per_img)
        last_imgs = imgs
        print(f'  Run {run+1}/{n_runs}: {per_img:.2f}s/img | Peak VRAM: {gpu_peak_memory_gb():.2f} GB')

    result = {
        'config': config.name, 'label': config.short_label(),
        'steps': config.num_inference_steps,
        'latency_mean_s': round(float(np.mean(latencies)), 3),
        'latency_std_s': round(float(np.std(latencies)), 3),
        'peak_vram_gb': round(gpu_peak_memory_gb(), 2),
    }

    # Save images to Drive
    for i, img in enumerate(last_imgs):
        img_dir = Path(f'results/images/{config.name}')
        img_dir.mkdir(parents=True, exist_ok=True)
        img.save(img_dir / f'sample_{i:02d}.png')

    del pipe; gc.collect(); torch.cuda.empty_cache()
    return result, last_imgs

print('✅ Benchmark function ready')

## 1. Profiling

In [ ]:
if not already_done('baseline'):
    baseline_cfg = CompressionConfig(name='baseline', dtype='fp16', num_inference_steps=50, guidance_scale=7.5)
    pipe = load_pipeline(baseline_cfg)
    gen = seed_everything(42)
    with timer() as t_enc:
        _ = pipe.encode_prompt(prompt='a test image', device='cuda')
    print(f'Text encoding: {t_enc["elapsed_s"]:.3f}s')

    reset_peak_memory()
    gen = seed_everything(42)
    with timer() as t_full:
        img = pipe('a test image', num_inference_steps=50, guidance_scale=7.5, generator=gen).images[0]
    print(f'\nBaseline: {t_full["elapsed_s"]:.2f}s | Peak VRAM: {gpu_peak_memory_gb():.2f} GB')
    del pipe; gc.collect(); torch.cuda.empty_cache()
else:
    print('Baseline already profiled')

## 2. Axis-by-Axis Compression

In [ ]:
# 2.1 Baseline
if not already_done('baseline'):
    cfg = CompressionConfig(name='baseline', dtype='fp16', num_inference_steps=50, guidance_scale=7.5)
    print(f'\n=== {cfg.name} ===')
    r, imgs = benchmark_one(cfg, prompts, n_runs=3)
    results.append(r); all_images[cfg.name] = imgs
    print(f'  → {r["latency_mean_s"]}s ± {r["latency_std_s"]}s | {r["peak_vram_gb"]} GB')
    save_checkpoint(results, '2.1_baseline')
else:
    print('✓ baseline done')

In [ ]:
# 2.2 Quantization
for cfg in [
    CompressionConfig(name='int8_unet', dtype='fp16', quantize_unet='int8', num_inference_steps=50, guidance_scale=7.5),
    CompressionConfig(name='nf4_unet', dtype='fp16', quantize_unet='nf4', num_inference_steps=50, guidance_scale=7.5),
]:
    if already_done(cfg.name):
        print(f'✓ {cfg.name} done'); continue
    print(f'\n=== {cfg.name} ===')
    r, imgs = benchmark_one(cfg, prompts, n_runs=3)
    results.append(r); all_images[cfg.name] = imgs
    print(f'  → {r["latency_mean_s"]}s ± {r["latency_std_s"]}s | {r["peak_vram_gb"]} GB')
    save_checkpoint(results, f'2.2_{cfg.name}')

In [ ]:
# 2.3 DeepCache
for cfg in [
    CompressionConfig(name='deepcache_2', dtype='fp16', use_deepcache=True, deepcache_interval=2, num_inference_steps=50, guidance_scale=7.5),
    CompressionConfig(name='deepcache_3', dtype='fp16', use_deepcache=True, deepcache_interval=3, num_inference_steps=50, guidance_scale=7.5),
]:
    if already_done(cfg.name):
        print(f'✓ {cfg.name} done'); continue
    print(f'\n=== {cfg.name} ===')
    r, imgs = benchmark_one(cfg, prompts, n_runs=3)
    results.append(r); all_images[cfg.name] = imgs
    print(f'  → {r["latency_mean_s"]}s ± {r["latency_std_s"]}s | {r["peak_vram_gb"]} GB')
    save_checkpoint(results, f'2.3_{cfg.name}')

In [ ]:
# 2.4 LCM-LoRA
for cfg in [
    CompressionConfig(name='lcm_8step', dtype='fp16', use_lcm_lora=True, num_inference_steps=8, guidance_scale=1.5),
    CompressionConfig(name='lcm_4step', dtype='fp16', use_lcm_lora=True, num_inference_steps=4, guidance_scale=1.5),
]:
    if already_done(cfg.name):
        print(f'✓ {cfg.name} done'); continue
    print(f'\n=== {cfg.name} ===')
    r, imgs = benchmark_one(cfg, prompts, n_runs=3)
    results.append(r); all_images[cfg.name] = imgs
    print(f'  → {r["latency_mean_s"]}s ± {r["latency_std_s"]}s | {r["peak_vram_gb"]} GB')
    save_checkpoint(results, f'2.4_{cfg.name}')

In [ ]:
# 2.5 torch.compile
if not already_done('compiled'):
    cfg = CompressionConfig(name='compiled', dtype='fp16', use_torch_compile=True, compile_mode='reduce-overhead', num_inference_steps=50, guidance_scale=7.5)
    print(f'\n=== {cfg.name} ===')
    r, imgs = benchmark_one(cfg, prompts, n_runs=3, n_warmup=3)
    results.append(r); all_images[cfg.name] = imgs
    print(f'  → {r["latency_mean_s"]}s ± {r["latency_std_s"]}s | {r["peak_vram_gb"]} GB')
    save_checkpoint(results, '2.5_compiled')
else:
    print('✓ compiled done')

In [ ]:
# 2.6 Tiny VAE
if not already_done('tiny_vae'):
    cfg = CompressionConfig(name='tiny_vae', dtype='fp16', use_tiny_vae=True, num_inference_steps=50, guidance_scale=7.5)
    print(f'\n=== {cfg.name} ===')
    r, imgs = benchmark_one(cfg, prompts, n_runs=3)
    results.append(r); all_images[cfg.name] = imgs
    print(f'  → {r["latency_mean_s"]}s ± {r["latency_std_s"]}s | {r["peak_vram_gb"]} GB')
    save_checkpoint(results, '2.6_tiny_vae')
else:
    print('✓ tiny_vae done')

## 3. Stacked Combinations

In [ ]:
combo_configs = [
    CompressionConfig(name='nf4+deepcache', dtype='fp16', quantize_unet='nf4',
                      use_deepcache=True, deepcache_interval=2,
                      num_inference_steps=50, guidance_scale=7.5),
    CompressionConfig(name='lcm+deepcache', dtype='fp16',
                      use_lcm_lora=True, use_deepcache=True, deepcache_interval=2,
                      num_inference_steps=8, guidance_scale=1.5),
    CompressionConfig(name='lcm+compiled+tvae', dtype='fp16',
                      use_lcm_lora=True, use_torch_compile=True,
                      use_tiny_vae=True, num_inference_steps=8, guidance_scale=1.5),
    # NOTE: full_stack with compile+deepcache crashes (CUDA graph conflict).
    # We use a no-compile version instead.
    CompressionConfig(name='full_stack_nocompile', dtype='fp16',
                      quantize_unet='nf4', use_lcm_lora=True,
                      use_deepcache=True, deepcache_interval=2,
                      use_tiny_vae=True,
                      num_inference_steps=8, guidance_scale=1.5),
]

for cfg in combo_configs:
    if already_done(cfg.name):
        print(f'✓ {cfg.name} done'); continue
    print(f'\n=== {cfg.name} ===')
    try:
        r, imgs = benchmark_one(cfg, prompts, n_runs=3, n_warmup=3)
        results.append(r); all_images[cfg.name] = imgs
        print(f'  → {r["latency_mean_s"]}s ± {r["latency_std_s"]}s | {r["peak_vram_gb"]} GB')
        save_checkpoint(results, f'3_{cfg.name}')
    except Exception as e:
        print(f'  ❌ FAILED: {e}')
        gc.collect(); torch.cuda.empty_cache()

## 4. Quality Evaluation (CLIP Score)

In [ ]:
df = pd.DataFrame(results)

# For configs with images in memory, compute CLIP
for cfg_name, imgs in all_images.items():
    print(f'CLIP score for {cfg_name}...')
    mean, std, scores = compute_clip_scores(imgs, prompts[:len(imgs)])
    df.loc[df['config'] == cfg_name, 'clip_score'] = mean
    df.loc[df['config'] == cfg_name, 'clip_score_std'] = std

# For configs from checkpoint (no images in memory), reload from Drive
missing = df[df['clip_score'].isna()]['config'].tolist()
if missing:
    print(f'\nReloading images from Drive for: {missing}')
    from PIL import Image
    for cfg_name in missing:
        img_dir = Path(f'results/images/{cfg_name}')
        if img_dir.exists():
            imgs = sorted(img_dir.glob('*.png'))
            if imgs:
                loaded = [Image.open(p) for p in imgs[:len(prompts)]]
                print(f'  CLIP score for {cfg_name} (from disk)...')
                mean, std, _ = compute_clip_scores(loaded, prompts[:len(loaded)])
                df.loc[df['config'] == cfg_name, 'clip_score'] = mean
                df.loc[df['config'] == cfg_name, 'clip_score_std'] = std
        else:
            print(f'  ⚠️ No images found for {cfg_name}')

# Speedup
baseline_lat = df.loc[df['config'] == 'baseline', 'latency_mean_s'].values
if len(baseline_lat) > 0:
    df['speedup'] = (baseline_lat[0] / df['latency_mean_s']).round(2)

print('\n' + '='*90)
display(df.sort_values('speedup', ascending=False) if 'speedup' in df.columns else df)

df.to_csv('results/benchmark_results.csv', index=False)
print('\n💾 Saved to Google Drive')

## 5. Visualizations

In [ ]:
if 'clip_score' in df.columns and df['clip_score'].notna().any():
    plot_pareto(df[df['clip_score'].notna()], 'results/pareto_frontier.png')
    display(IPImage('results/pareto_frontier.png', width=800))

plot_speedup_bar(df, output_path='results/speedup_bar.png')
display(IPImage('results/speedup_bar.png', width=700))

plot_memory_comparison(df, output_path='results/memory_comparison.png')
display(IPImage('results/memory_comparison.png', width=700))

if all_images:
    save_comparison_grid(all_images, prompts, 'results/comparison_grid.png', max_prompts=4)
    display(IPImage('results/comparison_grid.png', width=1000))

print('💾 All plots saved to Google Drive')

## 6. Pareto-Optimal Recommendations

In [ ]:
if 'clip_score' in df.columns and df['clip_score'].notna().any():
    pareto_df = find_pareto_frontier(df[df['clip_score'].notna()], 'latency_mean_s', 'clip_score')
    print('Pareto-optimal configurations:')
    display(pareto_df[['config', 'latency_mean_s', 'peak_vram_gb', 'clip_score', 'speedup']].sort_values('speedup', ascending=False))

print('\n📋 Deployment Recommendations:')
print('  🚀 Speed:    lcm_4step — 6.7× faster, no quality loss')
print('  ⚖️  Balanced: lcm+compiled+tvae — 5.5× faster, 25% less VRAM')
print('  🎨 Quality:  deepcache_2 — 1.7× faster, zero quality degradation')

## 7. Deployment Demo

In [ ]:
# Direct inference demo (more reliable than LitServe on free Colab)
cfg = CompressionConfig(name='speed_demo', dtype='fp16', use_lcm_lora=True,
                        num_inference_steps=4, guidance_scale=1.5)
pipe = load_pipeline(cfg)
gen = seed_everything(42)

# Warmup
_ = generate_images(pipe, cfg, ['warmup'], generator=seed_everything(0), height=512, width=512)

# Timed inference
gen = seed_everything(42)
t0 = time.perf_counter()
imgs = generate_images(pipe, cfg, ['a photo of an astronaut riding a horse on mars, cinematic lighting'], generator=gen)
latency = time.perf_counter() - t0

print(f'🚀 Optimized inference: {latency:.2f}s (vs ~5.7s baseline = {5.66/latency:.1f}× speedup)')
imgs[0].save('results/deployment_demo.png')
display(imgs[0])

del pipe; gc.collect(); torch.cuda.empty_cache()

## 8. Summary & Future Work

### Key Findings
- **Highest leverage:** LCM-LoRA step reduction (50 → 4 steps) gives 6.7× speedup with no quality loss
- **Best balanced:** LCM + torch.compile + TinyVAE — 5.5× faster, 25% less VRAM
- **Memory champion:** NF4 quantization saves ~3.4 GB but is slower on T4 (no INT4 tensor cores)
- **Negative results:** torch.compile has huge warmup variance on T4; DeepCache + CUDA graphs conflict; not all axes compose

### Ideas for Future Work
- Test on A100 where quantization speedup is expected to be positive
- Ternary quantization (BitNet-style) for the UNet
- Per-block mixed precision based on sensitivity analysis
- Speculative decoding for diffusion — small model for early steps
- Pruna integration — wrap all methods into `smash()` calls

In [ ]:
# Download results
!cd results && zip -r /content/results.zip .
from google.colab import files
files.download('/content/results.zip')